**Parcial II: Análisis de datos del mercado laboral GEIH 2026**

**Yaniz Dayana Molina Arango**

**1. Para el total nacional calcule (a Junio de 2026):**
- población en edad de trabajar
- población que no está en edad de trabajar
- población económicamente activa
- población económicamente inactiva
- tasa de ocupación
- tasa de desempleo

In [0]:
#Leyendo la tabla de características generales del catálogo
cgse=spark.read.format("delta").table("workspace.electiva.cgse")
ft = spark.read.format("delta").table("workspace.electiva.ft")

In [0]:
#Población en edad de trabajar 
PET = ft.filter(ft['PET'] == 1).agg({'FEX_C18': 'sum'}).collect()[0][0]
print(f'La población que está en edad de trabajar es de {PET:,.0f} personas')

Población en edad de trabajar (PET): 41.202.631 personas, esta cifra corresponde al total de personas que, de acuerdo con la edad establecida por las estadísticas, tienen la posibilidad de participar en el mercado laboral. Este grupo es importante porque permite estudiar las condiciones del empleo en el país. No obstante, esto no quiere decir que todas estas personas trabajen o estén interesadas en conseguir un empleo.

In [0]:
#Población que no esta en edad de trabajar
cgse = cgse.toPandas()
PT = cgse['FEX_C18'].sum()
PNET = PT - PET
print(f'La población que no está en edad de trabajar es de {PNET:,.0f} personas')

Población que no está en edad de trabajar: 11.346.181 personas, este grupo está integrado por las personas que aún no tienen la edad requerida para ser consideradas parte de la población en edad de trabajar, principalmente niños y niñas. Por esta razón, no se tienen en cuenta dentro de los cálculos relacionados con la participación laboral, la ocupación y el desempleo

In [0]:
#Población economicamente activa
PEA = ft.filter(ft['FT'] == 1).agg({'FEX_C18': 'sum'}).collect()[0][0]
print(f'La población económicamente activa es de {PEA:,.0f} personas')

Población económicamente activa (PEA): 26.593.429 personas, la población económicamente activa está formada por quienes cuentan con un empleo y por aquellas personas que no tienen trabajo, pero están buscando una oportunidad laboral y cumplen los requisitos para ser consideradas desempleadas. Este indicador ayuda a identificar cuántas personas están participando directamente en el mercado de trabajo, ya sea trabajando o intentando incorporarse a él.

In [0]:
#Población economicamente inactiva
PEI = PET - PEA
print(f'La población económicamente inactiva es de {PEI:,.0f} personas')

Población económicamente inactiva (PEI): 14.609.202 personas, la población económicamente inactiva incluye a las personas que tienen la edad necesaria para trabajar, pero que no cuentan con un empleo ni están buscando uno bajo los criterios establecidos. En este grupo se encuentran, por ejemplo, estudiantes, pensionados y personas que se dedican a las actividades del hogar. Es importante tener en cuenta que, aunque no participen en el mercado laboral, muchas de ellas realizan labores significativas para sus familias y la sociedad.

In [0]:
#Tasa de ocupación
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados").toPandas()
O = ocupados[ocupados['OCI']==1]['FEX_C18'].sum()
TO = O/PEA*100
TO = O/PET*100
print(f'El porcentaje de ocupados es de {TO:.2f}%')

Tasa de ocupación (TO): 59,36 %, este porcentaje muestra qué parte de la población en edad de trabajar tiene actualmente un empleo. En otras palabras, de cada 100 personas que cumplen con la edad requerida, aproximadamente 59 se encuentran trabajando. Este resultado permite observar el nivel de ocupación del país y conocer qué tan amplia es la participación de las personas en actividades laborales, aunque no refleja necesariamente las condiciones o la calidad de los empleos.

In [0]:
#Tasa de desempleo
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados").toPandas()
D = desempleados[desempleados['DSI']==1]['FEX_C18'].sum()  # Cell 23
TD = D/PEA*100
print(f'El porcentaje de desempleados es de {TD:.2f}%')

Tasa de desempleo (TD): 8,03 %, la tasa de desempleo señala el porcentaje de personas que forman parte de la población económicamente activa, pero que no tienen trabajo y cumplen las condiciones para ser clasificadas como desempleadas. De acuerdo con este resultado, cerca de 8 de cada 100 personas que participan en el mercado laboral se encuentran en esta situación. Esto demuestra que, a pesar de que muchas personas han logrado conseguir empleo, todavía existen dificultades para que todos los que buscan trabajar encuentren una oportunidad laboral.

**2. Calcule las tasas de desempleo para hombres y mujeres en todos los departamentos de Colombia a Junio de 2026, ¿cuáles son los departamentos con mayores y menores tasas de desempleo?**

In [0]:
import pandas as pd
# Leyendo la tabla de características generales
ft = spark.read.format("delta").table("workspace.electiva.ft")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()
# Primero hay que anexar la P3271 a la tabla de desempleados
desempleados = pd.merge(desempleados,cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')
ft = pd.merge(ft,cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],how='left')

In [0]:
# desempleados_dpto -> cantidad de desempleados por departamento
desempleados_dpto = desempleados[desempleados['DSI'] == 1].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()
desempleados_dpto

In [0]:
# Cantidad de hombres y mujeres económicamente activos por departamento
pea_dpto_genero = ft[ft['FT'] == 1].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()

In [0]:
tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero,'DESEMPLEADOS': desempleados_dpto})

In [0]:
tabla_desempleo
tabla_desempleo['TD'] = (tabla_desempleo['DESEMPLEADOS'] / tabla_desempleo['PEA']) * 100
tabla_desempleo.head(100)

In [0]:
# 4. Diccionarios de mapeo
dpto_nombre = {5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO',66: 'RISARALDA', 68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA',76: 'VALLE DEL CAUCA', 81: 'ARAUCA', 85: 'CASANARE', 86: 'PUTUMAYO',88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA', 95: 'GUAVIARE',97: 'VAUPES', 99: 'VICHADA'}
sexo = {1: 'Hombre', 2: 'Mujer'}
     

In [0]:
tabla_desempleo = tabla_desempleo.reset_index()
tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['P3271'].map(sexo)

In [0]:
tabla_desempleo = tabla_desempleo[['Departamento', 'Sexo', 'PEA', 'DESEMPLEADOS', 'TD']]

In [0]:
# Ordenar por dos variables de mayor a menor (descendente)
tabla_desempleo = tabla_desempleo.sort_values(by=['Sexo', 'TD'],ascending=False)
tabla_desempleo

In [0]:
tabla_desempleo[tabla_desempleo['Sexo'] == 'Mujer']

In [0]:
tabla_desempleo[tabla_desempleo['Sexo'] == 'Hombre']

Análisis de las tasas de desempleo por departamento: 
Al comparar las cifras de desempleo en los diferentes departamentos de Colombia, se pueden identificar diferencias importantes entre hombres y mujeres. En algunos territorios, las mujeres tienen mayores dificultades para encontrar trabajo que los hombres, mientras que en otros ocurre lo contrario. Esto permite reconocer que las condiciones laborales no son iguales en todo el país y que las oportunidades de conseguir empleo pueden cambiar dependiendo del lugar y del género.

Departamentos con mayores niveles de desempleo:
Entre los departamentos que presentan cifras más altas se encuentra Putumayo, donde el desempleo femenino llega al 25,02 %, mientras que el masculino se ubica en el 19,68 %. Por su parte, San Andrés muestra un comportamiento diferente, ya que los hombres registran una tasa del 20,98 %, superior al 18,16 % correspondiente a las mujeres.

De igual manera, Sucre presenta un porcentaje de desempleo femenino del 20,49 %, seguido de Bolívar, con un 17,40 %. En cuanto a los hombres, Arauca alcanza una tasa del 14,49 % y Guaviare registra un 12,13 %. Estas cifras reflejan que algunos departamentos enfrentan mayores dificultades para generar oportunidades laborales y que las diferencias entre ambos géneros dependen de las condiciones particulares de cada territorio.

Departamentos con menores niveles de desempleo:
Por otro lado, Vaupés presenta los porcentajes más bajos de desempleo entre los departamentos mencionados, con un 1,78 % para las mujeres y un 2,99 % para los hombres. Amazonas también muestra resultados reducidos, especialmente en la población femenina, cuya tasa corresponde al 2,61 %. Asimismo, Huila y Nariño se encuentran entre los territorios con niveles relativamente bajos de desempleo para ambos géneros.

En Risaralda se observa una diferencia considerable entre hombres y mujeres: la tasa de desempleo masculino es del 4,00 %, mientras que la femenina alcanza el 9,60 %. Esto demuestra que, incluso en departamentos con cifras generales más favorables, pueden existir diferencias importantes entre los grupos de población. Además, es necesario analizar estos resultados con cuidado, ya que una tasa baja de desempleo no significa necesariamente que existan mejores salarios, empleos estables o condiciones laborales adecuadas.

Conclusión general
En conclusión, los datos correspondientes a junio de 2026 permiten evidenciar que el desempleo en Colombia no afecta de la misma manera a todos los departamentos ni a hombres y mujeres por igual. Putumayo y San Andrés se destacan por presentar porcentajes elevados, aunque en cada uno las diferencias por género tienen un comportamiento distinto. En cambio, Vaupés registra las tasas más bajas dentro de los resultados analizados.

En términos generales, estas cifras muestran la importancia de estudiar el mercado laboral teniendo en cuenta las características de cada región. Para comprender mejor por qué existen estas diferencias, también sería conveniente revisar otros aspectos, como las principales actividades económicas de cada departamento, el trabajo informal, los niveles de ingresos y la disponibilidad de vacantes. De esta manera, se puede obtener una visión más completa de la situación laboral, sin asumir que un menor desempleo significa necesariamente mejores oportunidades de trabajo.




**3. Tomando en consideración únicamente los siguientes grupos de edad:**
- 18 a 24
- 24 a 30
- 30 a 40
- 40 a 50
- 60 en adelante

Para cada uno de los departamentos, cual es el grupo de edad con mayor y menor tasa de desempleo?

In [0]:
# Función para mapear la edad (P6040) a grupos de edad
def mapear_grupo_edad(edad):
    if 18 <= edad < 24:
        return '18 a 24'
    elif 24 <= edad < 30:
        return '24 a 30'
    elif 30 <= edad < 40:
        return '30 a 40'
    elif 40 <= edad < 50:
        return '40 a 50'
    elif edad >= 60:
        return '60 en adelante'
    else:return None

# Aplicar el mapeo a ambos DataFrames
desempleados['GRUPO_EDAD'] = desempleados['P6040'].map(mapear_grupo_edad)
ft['GRUPO_EDAD'] = ft['P6040'].map(mapear_grupo_edad)

In [0]:
# Desempleados por grupo de edad y departamento (reutilizando código de pregunta 2)
desempleados_dpto_edad = desempleados[desempleados['DSI'] == 1].groupby(['GRUPO_EDAD', 'DPTO'])['FEX_C18'].sum()
# PEA por grupo de edad y departamento
pea_dpto_edad = ft[ft['FT'] == 1].groupby(['GRUPO_EDAD', 'DPTO'])['FEX_C18'].sum()
# Crear tabla de desempleo (reutilizando estructura de pregunta 2)
tabla_desempleo_edad = pd.DataFrame({'PEA': pea_dpto_edad, 'DESEMPLEADOS': desempleados_dpto_edad})

In [0]:
# Calcular tasa de desempleo (reutilizando código de pregunta 2)
tabla_desempleo_edad['TD'] = (tabla_desempleo_edad['DESEMPLEADOS'] / tabla_desempleo_edad['PEA']) * 100
# Mapear nombres de departamentos (reutilizando dpto_nombre de pregunta 2)
tabla_desempleo_edad = tabla_desempleo_edad.reset_index()
tabla_desempleo_edad['Departamento'] = tabla_desempleo_edad['DPTO'].map(dpto_nombre)
# Seleccionar y ordenar columnas
tabla_desempleo_edad = tabla_desempleo_edad[['Departamento', 'GRUPO_EDAD', 'PEA', 'DESEMPLEADOS', 'TD']]
tabla_desempleo_edad = tabla_desempleo_edad.sort_values(by=['Departamento', 'TD'], ascending=[True, False])

display(tabla_desempleo_edad)

In [0]:
# Grupo de edad con mayor tasa de desempleo por departamento
max_td = tabla_desempleo_edad.loc[tabla_desempleo_edad.groupby('Departamento')['TD'].idxmax()]
max_td = max_td[['Departamento', 'GRUPO_EDAD', 'TD']].reset_index(drop=True)
max_td.columns = ['Departamento', 'Mayor grupo de edad', 'Mayor TD']
# Grupo de edad con menor tasa de desempleo por departamento
min_td = tabla_desempleo_edad.loc[tabla_desempleo_edad.groupby('Departamento')['TD'].idxmin()]
min_td = min_td[['Departamento', 'GRUPO_EDAD', 'TD']].reset_index(drop=True)
min_td.columns = ['Departamento', 'Menor grupo de edad', 'Menor TD']
# Combinar resultados
resultado_edad = pd.merge(max_td, min_td, on='Departamento')
display(resultado_edad)

Análisis del desempleo según la edad y el departamento:
Al estudiar las cifras de desempleo de acuerdo con la edad y el departamento, se puede observar que las posibilidades de conseguir trabajo cambian entre los distintos grupos de población. En general, los jóvenes de 18 a 24 años son quienes presentan mayores dificultades para encontrar empleo en la mayoría de las regiones analizadas. En cambio, las personas de 60 años o más muestran porcentajes de desempleo más bajos. Estas diferencias permiten comprender que la edad puede estar relacionada con las oportunidades de ingresar y permanecer en el mundo laboral.

Departamentos con mayores tasas de desempleo por edad:
Entre los resultados que más llaman la atención se encuentra San Andrés, donde el desempleo de las personas entre 18 y 24 años alcanza el 33,66 %. Una situación similar se presenta en Sucre, con un 33,14 % para este mismo grupo. Esto quiere decir que cerca de una tercera parte de los jóvenes económicamente activos de estas zonas no cuenta con un empleo.

De igual manera, Vichada registra una tasa del 31,60 % y Bolívar presenta un 30,86 % entre los jóvenes de 18 a 24 años. Por otro lado, en Putumayo, el porcentaje más alto corresponde a las personas de 24 a 30 años, con un 30,58 %. Estas cifras reflejan los obstáculos que pueden encontrar los jóvenes al momento de comenzar su vida laboral, especialmente cuando tienen poca experiencia o deben competir por las vacantes disponibles. Sin embargo, para conocer las razones específicas de estos resultados, sería necesario considerar las características económicas de cada departamento y las condiciones que exigen las empresas para contratar personal.

Departamentos con menores tasas de desempleo por edad:
En cuanto a los porcentajes más bajos, estos se encuentran principalmente en la población de 60 años en adelante. Risaralda registra una tasa del 0,33 %, seguido de Meta con el 1,18 %, Chocó con el 1,21 %, Cauca con el 1,27 % y Magdalena con el 1,40 %.

Estos resultados pueden estar relacionados, en parte, con el hecho de que algunas personas mayores dejan de buscar empleo y pasan a formar parte de la población económicamente inactiva. Esto es importante porque la tasa de desempleo solo incluye a quienes participan en el mercado laboral y cumplen los requisitos para ser considerados desempleados. Por lo tanto, tener un porcentaje reducido no significa necesariamente que las personas mayores tengan más facilidades para trabajar o que sus condiciones laborales sean mejores.

Diferencias entre los grupos de edad:
Los datos analizados muestran que los jóvenes de 18 a 24 años tienen las tasas de desempleo más elevadas en 25 de los 33 departamentos estudiados. Por su parte, las personas entre 24 y 30 años presentan los mayores porcentajes en siete departamentos, entre los que se encuentran Amazonas, Arauca, Boyacá, Chocó, Nariño, Putumayo y Vaupés.

Una muestra de estas diferencias se encuentra en Bolívar, donde el desempleo para las personas de 18 a 24 años es del 30,86 %, mientras que en el grupo de 60 años o más corresponde al 2,98 %. Esto representa una diferencia de 27,88 puntos porcentuales entre ambos grupos. Este contraste permite identificar que el desempleo no afecta de igual manera a todas las edades y que los jóvenes enfrentan mayores dificultades para incorporarse al mercado laboral en buena parte del territorio.

Conclusión general
En conclusión, las cifras permiten reconocer que la situación del desempleo en Colombia varía tanto por la edad de las personas como por el departamento en el que se encuentran. Los jóvenes de 18 a 24 años concentran los mayores niveles de desempleo en la mayoría de las regiones estudiadas, mientras que las personas de 60 años o más presentan porcentajes más reducidos. Esto evidencia la importancia de promover alternativas que faciliten el acceso de los jóvenes al empleo, como oportunidades para adquirir experiencia y mayores posibilidades de contratación.

Asimismo, es fundamental tener en cuenta las particularidades económicas de cada departamento para comprender mejor estas diferencias. También se debe considerar que las estadísticas de desempleo no muestran toda la realidad laboral, pues no reflejan por sí solas aspectos como los salarios, la estabilidad, la informalidad o la situación de quienes han dejado de buscar trabajo. Por esta razón, analizar estos factores en conjunto permite tener una perspectiva más amplia de las condiciones laborales del país.




**4. ¿Cuáles son los 3 departamentos con menor tasa de desempleo de mujeres entre los 18 y los 25 años ?**

In [0]:
# Mujeres desempleadas entre 18 y 25 años por departamento
desempleadas_jovenes = desempleados[(desempleados['DSI'] == 1) & (desempleados['P3271'] == 2) & (desempleados['P6040'].between(18, 25))].groupby('DPTO')['FEX_C18'].sum()

# Mujeres económicamente activas entre 18 y 25 años por departamento
pea_mujeres_jovenes = ft[(ft['FT'] == 1) & (ft['P3271'] == 2) & (ft['P6040'].between(18, 25))].groupby('DPTO')['FEX_C18'].sum()

# Crear tabla con los resultados
tabla_mujeres_jovenes = pd.DataFrame({'PEA': pea_mujeres_jovenes,'DESEMPLEADAS': desempleadas_jovenes})

# Completar con cero donde no se registraron desempleadas
tabla_mujeres_jovenes['DESEMPLEADAS'] = (tabla_mujeres_jovenes['DESEMPLEADAS'].fillna(0))

# Conservar departamentos con población económicamente activa
tabla_mujeres_jovenes = tabla_mujeres_jovenes[tabla_mujeres_jovenes['PEA'] > 0].copy()

# Calcular la tasa de desempleo
tabla_mujeres_jovenes['TD'] = (tabla_mujeres_jovenes['DESEMPLEADAS'] /tabla_mujeres_jovenes['PEA']) * 100

# Agregar los nombres de los departamentos
tabla_mujeres_jovenes = tabla_mujeres_jovenes.reset_index()
tabla_mujeres_jovenes['Departamento'] = (tabla_mujeres_jovenes['DPTO'].map(dpto_nombre))

# Excluir Bogotá, porque es un distrito y se solicitan departamentos
tabla_mujeres_jovenes = tabla_mujeres_jovenes[tabla_mujeres_jovenes['DPTO'] != 11]

# Seleccionar los tres departamentos con menor tasa de desempleo
resultado_punto4 = tabla_mujeres_jovenes.nsmallest(3, 'TD')[['Departamento', 'PEA', 'DESEMPLEADAS', 'TD']].reset_index(drop=True)

display(resultado_punto4)

# Respuesta escrita con los resultados obtenidos
print('Los tres departamentos con menor tasa de desempleo ''de mujeres entre 18 y 25 años son:')

for _, fila in resultado_punto4.iterrows():print(f"- {fila['Departamento']}: {fila['TD']:.2f}%.")

print('\nEstas tasas representan el porcentaje de mujeres desocupadas ''dentro de las mujeres económicamente activas de 18 a 25 años ''en cada departamento. Una tasa menor no implica necesariamente ''mayor participación laboral ni mejores condiciones de empleo.')
     

En conclusión, Vaupés, Nariño y Amazonas presentan las tasas más bajas de desempleo en mujeres de 18 a 25 años, destacándose Vaupés con un 0,00 %. Estos resultados indican que una menor proporción de las mujeres jóvenes que participan en el mercado laboral se encuentra desempleada en estos departamentos. Sin embargo, es importante tener en cuenta que estas cifras no garantizan mejores oportunidades laborales ni condiciones de trabajo, por lo que deben analizarse junto con otros indicadores del mercado laboral.

**5. Calcule el promedio de ingresos laborales (variable INGLABO) de las personas residentes en los departamentos de Risaralda, Caldas , Quindío, Antioquia y Valle del Cauca.**

In [0]:
# Leer la tabla de ocupados
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados").toPandas()

# Departamentos: Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca
departamentos = [66, 17, 63, 5, 76]

# Seleccionar personas ocupadas de los departamentos solicitados
ingresos_dpto = ocupados[(ocupados['OCI'] == 1) &(ocupados['DPTO'].isin(departamentos))].copy()

# Convertir ingresos y factor de expansión a valores numéricos
ingresos_dpto['INGLABO'] = pd.to_numeric(ingresos_dpto['INGLABO'], errors='coerce')

ingresos_dpto['FEX_C18'] = pd.to_numeric(ingresos_dpto['FEX_C18'], errors='coerce')

# Conservar ingresos registrados y factores positivos
ingresos_dpto = ingresos_dpto[ingresos_dpto['INGLABO'].notna() &(ingresos_dpto['FEX_C18'] > 0)].copy()

# Calcular el ingreso ponderado
ingresos_dpto['INGRESO_PONDERADO'] = (ingresos_dpto['INGLABO'] * ingresos_dpto['FEX_C18'])

# Sumar los ingresos ponderados y los factores por departamento
tabla_ingresos = ingresos_dpto.groupby('DPTO')[['INGRESO_PONDERADO', 'FEX_C18']].sum().reset_index()

# Calcular el promedio de ingresos laborales
tabla_ingresos['Promedio INGLABO'] = (tabla_ingresos['INGRESO_PONDERADO'] /tabla_ingresos['FEX_C18'])

# Agregar nombres de departamentos
tabla_ingresos['Departamento'] = (tabla_ingresos['DPTO'].map(dpto_nombre))

# Seleccionar columnas y ordenar de mayor a menor ingreso
resultado_punto5 = tabla_ingresos[['Departamento', 'Promedio INGLABO']].sort_values(by='Promedio INGLABO', ascending=False).reset_index(drop=True)

display(resultado_punto5)


En conclusión, al comparar los ingresos laborales promedio de Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca, se observa que Antioquia presenta el valor más alto, con aproximadamente $2.465.235, mientras que Valle del Cauca registra el promedio más bajo, con cerca de $2.122.050. Por su parte, Caldas, Risaralda y Quindío muestran resultados intermedios. Estas diferencias permiten identificar variaciones en los ingresos laborales entre los departamentos analizados. Sin embargo, para comprender mejor esta situación, sería necesario considerar otros factores, como el tipo de empleo, el nivel educativo, la actividad económica y las condiciones laborales de cada región.

**6. Para los mismos departamentos del punto anterior, calcule el promedio de ingresos laborales de hombres y mujeres en los siguientes rangos de edad :**
- 8 a 24
- 24 a 30
- 30 a 40
- 40 a 50
- 60 en adelante

In [0]:
import numpy as np
# Anexar P3271 (sexo) y P6040 (edad) desde cgse a ocupados
# Limpiar columnas previas para evitar sufijos _x/_y al re-ejecutar
ocupados = ocupados.loc[:, ~ocupados.columns.str.match(r'^(P3271|P6040)(_x|_y)?$')]
ocupados = pd.merge(ocupados,cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],how='left')

# Departamentos: Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca
departamentos = [66, 17, 63, 5, 76]

# Seleccionar personas ocupadas
ingresos_edad = ocupados[(ocupados['OCI'] == 1) & (ocupados['DPTO'].isin(departamentos))].copy()

# Verificar y convertir las variables a valores numéricos
for columna in ['INGLABO', 'FEX_C18', 'P6040', 'P3271']:ingresos_edad[columna] = pd.to_numeric(ingresos_edad[columna], errors='coerce')

# Filtrar datos válidos
ingresos_edad = ingresos_edad[ingresos_edad['INGLABO'].notna() & ingresos_edad['FEX_C18'].notna() & (ingresos_edad['FEX_C18'] > 0) & ingresos_edad['P6040'].notna() & ingresos_edad['P3271'].isin([1, 2])].copy()

# Clasificar los grupos de edad
def mapear_grupo_edad(edad):
    if 18 <= edad < 24:
        return '18 a 24'
    elif 24 <= edad < 30:
        return '24 a 30'
    elif 30 <= edad < 40:
        return '30 a 40'
    elif 40 <= edad < 50:
        return '40 a 50'
    elif edad >= 60:
        return '60 en adelante'
    else:
        return None

ingresos_edad['GRUPO_EDAD'] = (ingresos_edad['P6040'].apply(mapear_grupo_edad))

# Identificar hombres y mujeres
ingresos_edad['Sexo'] = ingresos_edad['P3271'].map({1: 'Hombre',2: 'Mujer'})

# Excluir edades fuera de los grupos solicitados
ingresos_edad = ingresos_edad[ingresos_edad['GRUPO_EDAD'].notna()].copy()

# Calcular el ingreso ponderado
ingresos_edad['INGRESO_PONDERADO'] = (ingresos_edad['INGLABO'] * ingresos_edad['FEX_C18'])

# Agrupar por departamento, edad y sexo
tabla_ingresos_edad = ingresos_edad.groupby(['DPTO', 'GRUPO_EDAD', 'Sexo'])[['INGRESO_PONDERADO', 'FEX_C18']].sum().reset_index()

# Calcular el promedio ponderado
tabla_ingresos_edad['Promedio INGLABO'] = (tabla_ingresos_edad['INGRESO_PONDERADO'] /tabla_ingresos_edad['FEX_C18'].replace(0, np.nan))

# Agregar los nombres de los departamentos
tabla_ingresos_edad['Departamento'] = (tabla_ingresos_edad['DPTO'].map(dpto_nombre))

# Organizar y mostrar los resultados
resultado_punto6 = tabla_ingresos_edad[['Departamento', 'GRUPO_EDAD', 'Sexo', 'Promedio INGLABO']].sort_values(by=['Departamento', 'GRUPO_EDAD', 'Sexo']).reset_index(drop=True)

display(resultado_punto6)

Análisis de los ingresos laborales según la edad, el sexo y el departamento:
Al comparar los ingresos laborales promedio de hombres y mujeres en Antioquia, Caldas, Quindío, Risaralda y Valle del Cauca, se pueden identificar diferencias importantes según la edad, el sexo y el lugar de residencia. En la mayoría de los grupos analizados, los hombres presentan ingresos promedio superiores a los de las mujeres. Sin embargo, existen algunas excepciones en las que las mujeres reciben ingresos más altos, lo que demuestra que estas diferencias no se comportan de la misma manera en todos los departamentos.

Diferencias de ingresos entre hombres y mujeres:
En los datos se observa que los hombres tienen mayores ingresos promedio en gran parte de los rangos de edad estudiados. Esta diferencia se evidencia especialmente en Antioquia, donde los hombres de 30 a 40 años registran un ingreso promedio de aproximadamente $2.960.273, mientras que las mujeres del mismo grupo reciben cerca de $2.300.981.

No obstante, también se presentan situaciones diferentes. En Quindío, las mujeres de 24 a 30 años tienen un ingreso promedio de $2.053.682, ligeramente superior al de los hombres, que corresponde a $2.026.891. Algo similar ocurre en Risaralda, donde las mujeres de este mismo rango de edad alcanzan un promedio de $2.566.475, frente a $1.954.558 en los hombres. Estos resultados reflejan que la diferencia de ingresos entre ambos sexos puede variar de acuerdo con la edad y las características laborales de cada territorio.

Comportamiento de los ingresos según la edad:
Al observar los grupos de edad, se encuentra que las personas entre los 30 y 50 años suelen registrar ingresos promedio más elevados en comparación con los grupos más jóvenes. Esto puede estar relacionado con factores como la experiencia adquirida, la trayectoria profesional y el acceso a determinados puestos de trabajo.

Por el contrario, en el grupo de 18 a 24 años los ingresos tienden a ser menores, posiblemente porque algunas personas se encuentran iniciando su vida laboral y todavía están adquiriendo experiencia. En el caso de las personas de 60 años en adelante, también se observan ingresos inferiores a los de varios grupos de edad intermedia, aunque el comportamiento cambia según el departamento y el sexo.

Comparación entre departamentos:
En cuanto a las diferencias territoriales, Antioquia sobresale por presentar algunos de los ingresos promedio más altos, especialmente entre los hombres de 30 a 40 años. Por su parte, Caldas, Quindío y Risaralda muestran variaciones entre los diferentes grupos, con resultados particulares en ciertos rangos de edad. En Valle del Cauca, varios grupos presentan ingresos promedio inferiores a los registrados en Antioquia, especialmente entre las mujeres.

También se debe destacar que las diferencias no son iguales para todas las edades. Por ejemplo, mientras que los hombres de Antioquia entre 30 y 40 años alcanzan un promedio cercano a los $2.960.273, las mujeres de Valle del Cauca de 60 años en adelante registran aproximadamente $1.015.907. Esta comparación permite observar una diferencia considerable entre los ingresos de ambos grupos, aunque también intervienen factores como la edad y el departamento de residencia.

Conclusión general
En conclusión, los resultados muestran que los ingresos laborales promedio presentan diferencias importantes dependiendo del sexo, la edad y el departamento. En la mayoría de los casos, los hombres reciben ingresos superiores a los de las mujeres, aunque existen excepciones en Quindío y Risaralda, especialmente entre las personas de 24 a 30 años. Asimismo, los grupos de edad intermedia suelen registrar ingresos más elevados, mientras que los jóvenes y las personas mayores presentan valores más bajos en varios de los departamentos analizados.

Estos resultados permiten comprender mejor las diferencias económicas que existen entre los grupos de población. Sin embargo, para explicar sus causas es necesario considerar otros elementos, como el nivel educativo, la experiencia laboral, el sector económico, el tipo de contrato y las horas trabajadas. De esta manera, se puede obtener una visión más completa de las condiciones de ingreso de hombres y mujeres en las regiones estudiadas.

**7. ¿Cuáles son los oficios con mayores promedios salariales a nivel nacional en general?**

In [0]:
# 1. Seleccionar todas las personas ocupadas del país
ingresos_nacional = ocupados[ocupados['OCI'] == 1].copy()

# 2. Convertir ingresos y factor de expansión a valores numéricos
for columna in ['INGLABO', 'FEX_C18']:ingresos_nacional[columna] = pd.to_numeric(ingresos_nacional[columna], errors='coerce')

# 3. Filtrar los datos válidos
ingresos_nacional = ingresos_nacional[ingresos_nacional['INGLABO'].notna() & ingresos_nacional['FEX_C18'].notna() &
(ingresos_nacional['FEX_C18'] > 0) & ingresos_nacional['OFICIO_C8'].notna()].copy()

# 4. Calcular el ingreso ponderado
ingresos_nacional['INGRESO_PONDERADO'] = (ingresos_nacional['INGLABO'] *ingresos_nacional['FEX_C18'])

# 5. Agrupar por código de oficio
tabla_oficios = ingresos_nacional.groupby('OFICIO_C8')[['INGRESO_PONDERADO', 'FEX_C18']].sum().reset_index()

# 6. Calcular el promedio ponderado
tabla_oficios['Promedio salarial'] = (tabla_oficios['INGRESO_PONDERADO'] /tabla_oficios['FEX_C18'].replace(0, np.nan))

# 7. Mostrar los 10 oficios con mayores ingresos promedio
resultado_punto7 = tabla_oficios.sort_values(by='Promedio salarial',ascending=False).head(10).reset_index(drop=True)

display(resultado_punto7)

Análisis de los oficios con mayores ingresos promedio a nivel nacional
Los resultados obtenidos permiten identificar que existen diferencias considerables en los ingresos laborales promedio de acuerdo con el oficio que desempeñan las personas en Colombia. Según los datos, el código 1111 presenta el promedio salarial más alto, con aproximadamente $28.990.595, seguido del código 1120, con cerca de $13.709.381, y del código 2212, con alrededor de $13.168.474. Estas cifras muestran que algunas ocupaciones alcanzan niveles de remuneración mucho más elevados que otras.

Diferencias salariales entre las ocupaciones
Entre los demás resultados también se destacan los códigos 2612, con un ingreso promedio de $12.330.042; 1223, con $9.914.578, y 1330, con $9.640.736. Por su parte, las ocupaciones identificadas con los códigos 2422, 2522, 1323 y 3321 presentan promedios que oscilan entre aproximadamente $7.973.173 y $8.622.638. Esto permite observar que, incluso dentro del grupo de ocupaciones con mayores ingresos, existen diferencias importantes en los niveles de remuneración.

Conclusión general
En conclusión, los datos evidencian que los ingresos laborales en Colombia varían considerablemente según la ocupación ejercida. El código 1111 sobresale por registrar el promedio salarial más elevado, mientras que los demás oficios presentan valores inferiores. Estas diferencias podrían estar relacionadas con aspectos como las responsabilidades del cargo, la experiencia, la formación académica y el nivel de especialización requerido. Sin embargo, para interpretar correctamente los resultados, es importante tener en cuenta que se trata de promedios ponderados estimados a partir de la GEIH, por lo que no representan necesariamente el salario que recibe cada trabajador de manera individual.